# cells

> The SolveIt-style cell GUI: a stack of typed cells (code / note / prompt / raw),
each toggleable in/out of the LLM's view. Because the model call is stateless,
context is just re-assembled from whichever cells are currently visible.

Note cells are markdown (KaTeX math, images, raw HTML). Standard Jupyter
command-mode hotkeys drive selection and editing.

In [ ]:
#| default_exp cells

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, shutil, subprocess, sys, threading, time
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
import fasthtml.components as fh
from fasthtml.svg import Path as SvgPath  # fastcore's pathlib.Path shadows fasthtml's svg <path> element otherwise
from toolslm.shell import get_shell
from datetime import datetime
import nbformat as _nbf
from lisette import *

## DaisyUI + app setup

CDN headers for DaisyUI + Tailwind, plus `KatexMarkdownJS()` (renders `.marked`
elements as markdown + KaTeX) and the command-mode hotkey listener.

In [ ]:
#| export
# Jupyter-style command-mode hotkeys. Fire only when no textarea/input is focused;
# each maps to an htmx POST that re-renders #notebook.
_HOTKEYS_JS = r"""
(function(){
  let lastD = 0;
  const act = (url) => htmx.ajax('POST', url, {target:'#notebook', swap:'outerHTML'});

  // Toggle '# ' comments on the selected lines of a textarea (Cmd/Ctrl+/).
  const toggleComment = (ta) => {
    const v = ta.value, s = ta.selectionStart, e = ta.selectionEnd;
    const ls = v.lastIndexOf('\n', s - 1) + 1;
    let le = v.indexOf('\n', e); if (le === -1) le = v.length;
    const lines = v.slice(ls, le).split('\n');
    const commented = lines.every(l => l.trim() === '' || l.trimStart().startsWith('#'));
    const out = commented
      ? lines.map(l => l.replace(/^(\s*)#\s?/, '$1')).join('\n')
      : lines.map(l => l.trim() === '' ? l : l.replace(/^(\s*)/, '$1# ')).join('\n');
    ta.value = v.slice(0, ls) + out + v.slice(le);
    ta.selectionStart = ls; ta.selectionEnd = ls + out.length;
  };

  document.addEventListener('keydown', (e) => {
    const a = document.activeElement;
    if (a && a.closest && a.closest('.CodeMirror')) return;  // CodeMirror handles its own keys
    const inEditor = a && (a.tagName === 'TEXTAREA' || a.tagName === 'INPUT' || a.isContentEditable);
    const mod = e.metaKey || e.ctrlKey;

    // --- editor shortcuts (fire while typing) ---
    if (mod && !e.shiftKey && e.key === '/') {
      if (a && a.tagName === 'TEXTAREA') { toggleComment(a); e.preventDefault(); }
      return;
    }
    if (mod && e.shiftKey && (e.code === 'Minus' || e.key === '-' || e.key === '_')) {
      if (a && a.id === 'compose-input') {
        htmx.ajax('POST', '/split', {target:'#app', swap:'outerHTML',
                   values:{source: a.value, pos: a.selectionStart}});
        e.preventDefault();
      }
      return;
    }

    // --- command-mode shortcuts (only when not editing) ---
    if (inEditor || e.metaKey || e.ctrlKey || e.altKey) return;
    let handled = true;
    switch (e.key) {
      case 'a': act('/insert?where=above'); break;
      case 'b': act('/insert?where=below'); break;
      case 'j': case 'ArrowDown': act('/select_delta?delta=1');  break;
      case 'k': case 'ArrowUp':   act('/select_delta?delta=-1'); break;
      case 'm': act('/settype_selected?t=note'); break;
      case 'y': act('/settype_selected?t=code'); break;
      case 'r': act('/settype_selected?t=raw');  break;
      case 's': boopSaveNotebook(); break;
      case 'x': act('/cut_selected'); break;
      case 'c': htmx.ajax('POST', '/copy_selected', {swap:'none'}); break;
      case 'v': act('/paste_selected'); break;
      case 'd': { const n = Date.now();
        if (n - lastD < 500) { lastD = 0; act('/del_selected'); }
        else { lastD = n; handled = false; }
        break; }
      default: handled = false;
    }
    if (handled) e.preventDefault();
  });
})();
"""

In [ ]:
#| export
# Tailwind's reset flattens markdown headings/lists; restore them for `.marked` content.
_MARKED_CSS = """
.marked h1{font-size:1.6rem;font-weight:700;margin:.4em 0}
.marked h2{font-size:1.35rem;font-weight:700;margin:.4em 0}
.marked h3{font-size:1.15rem;font-weight:600;margin:.4em 0}
.marked h4{font-size:1.05rem;font-weight:600;margin:.4em 0}
.marked ul{list-style:disc;margin:.3em 0 .3em 1.5rem}
.marked ol{list-style:decimal;margin:.3em 0 .3em 1.5rem}
.marked p{margin:.4em 0}
.marked a{color:#2563eb;text-decoration:underline}
.marked code{background:rgba(127,127,127,.2);padding:.1em .3em;border-radius:.25rem;font-family:monospace;color:var(--color-secondary)}
.marked pre{background:rgba(127,127,127,.15);padding:.6em;border-radius:.4rem;overflow:auto}
.marked pre code{background:none;padding:0;color:inherit}
.marked blockquote{border-left:3px solid #999;padding-left:.75em;margin:.4em 0;opacity:.85}
.marked table{border-collapse:collapse}
.marked th,.marked td{border:1px solid #999;padding:.2em .5em}
.marked img{max-width:100%}
.CodeMirror{height:auto;border:1px solid #ccc;border-radius:.4rem;font-size:.85rem}
.CodeMirror-scroll{max-height:60vh}
"""

In [ ]:
#| export
_EDIT_JS = r"""
function boopSave(id){
  var cm = window['_boopcm_'+id];
  if(cm) cm.save();
  window._boopFocusAfter = id;
  var ta = document.getElementById('ta-'+id);
  if(ta && ta.form) ta.form.requestSubmit();
}
function boopComposerSubmit(){
  var ta = document.getElementById('compose-input');
  var cm = window._boopComposerCM;
  if(cm && cm.getWrapperElement && cm.getWrapperElement().isConnected) cm.save();
  if(ta && ta.form) ta.form.requestSubmit();
}
function boopRestartServer(){
  if(!confirm('Restart boopiter? Unsaved notebook changes will be lost -- Save first if you want to keep them.')) return;
  fetch('/restart_server', {method:'POST'}).catch(function(){});
  var tries = 0;
  var poll = setInterval(function(){
    tries++;
    fetch('/_boopiter_ping').then(function(r){ if(r.ok){ clearInterval(poll); location.reload(); } }).catch(function(){});
    if(tries > 60) clearInterval(poll);  // ~30s safety timeout
  }, 500);
}
function boopSyncAllEditors(){
  // Flush every currently-open editor's text to the server (no execution) before Save writes to disk --
  // otherwise an edit that was never Shift-Entered would silently vanish, unlike real Jupyter's WYSIWYG save.
  var jobs = [];
  (window._boopcms || []).forEach(function(cm){
    cm.save();
    var id = cm.getTextArea().getAttribute('data-cid');
    if(id) jobs.push(fetch('/sync_cell?id='+id, {method:'POST',
      headers:{'Content-Type':'application/x-www-form-urlencoded'},
      body:'source='+encodeURIComponent(cm.getValue())}));
  });
  document.querySelectorAll('textarea[data-cm="edit"]').forEach(function(ta){
    var id = ta.getAttribute('data-cid');
    if(id) jobs.push(fetch('/sync_cell?id='+id, {method:'POST',
      headers:{'Content-Type':'application/x-www-form-urlencoded'},
      body:'source='+encodeURIComponent(ta.value)}));
  });
  return Promise.all(jobs);
}
function boopSaveNotebook(){
  boopSyncAllEditors().then(function(){ htmx.ajax('POST', '/save_now', {swap:'none'}); });
}
function boopComposerSplit(cm){
  htmx.ajax('POST', '/split', {target:'#notebook', swap:'beforeend',
    values:{source: cm.getValue(), pos: cm.indexFromPos(cm.getCursor())}});
}
function boopMakeCM(ta, isComposer){
  var dark = window._boopDark !== false;
  var id = ta.getAttribute('data-cid');
  var extra = isComposer ? {
      'Shift-Enter': boopComposerSubmit, 'Ctrl-Enter': boopComposerSubmit, 'Cmd-Enter': boopComposerSubmit,
      'Ctrl-/': function(cm){ cm.toggleComment(); }, 'Cmd-/': function(cm){ cm.toggleComment(); },
      'Shift-Ctrl--': function(cm){ boopComposerSplit(cm); }, 'Shift-Cmd--': function(cm){ boopComposerSplit(cm); }
    } : {
      'Shift-Enter': function(){ boopSave(id); }, 'Ctrl-Enter': function(){ boopSave(id); }, 'Cmd-Enter': function(){ boopSave(id); },
      'Ctrl-/': function(cm){ cm.toggleComment(); }, 'Cmd-/': function(cm){ cm.toggleComment(); }
    };
  var cm = CodeMirror.fromTextArea(ta, {
    mode:'python', theme: dark?'material-darker':'default',
    lineNumbers:true, lineWrapping:false, viewportMargin:Infinity, indentUnit:4, extraKeys: extra
  });
  if(isComposer){ window._boopComposerCM = cm; cm.focus(); }
  else {
    window['_boopcm_'+id] = cm; (window._boopcms = window._boopcms || []).push(cm);
    if(String(window._boopFocusAfter) === String(id)){
      window._boopFocusAfter = null; cm.focus(); cm.setCursor(cm.lineCount(), 0);
    }
  }
  return cm;
}
function boopCopy(id, ctype){
  var cm = window['_boopcm_'+id];
  var text;
  if(ctype === 'code' && cm){ text = cm.getValue(); }
  else {
    var btn = document.getElementById('copy-'+id);
    text = btn ? (btn.getAttribute('data-src') || '') : '';
  }
  navigator.clipboard.writeText(text).catch(function(){});
}
function boopRenderAnsi(root){
  if(!window.AnsiUp) return;  // module still loading; a later retry will pick these up
  var scope = (root && root.querySelectorAll) ? root : document;
  scope.querySelectorAll('.ansi-out:not([data-ansi-done])').forEach(function(el){
    el.setAttribute('data-ansi-done', '1');
    var au = new window.AnsiUp(); au.use_classes = false;
    el.innerHTML = au.ansi_to_html(el.textContent);
  });
}
function boopInitEditors(root){
  var scope = (root && root.querySelectorAll) ? root : document;
  scope.querySelectorAll('textarea[data-cm]:not([data-cminit])').forEach(function(ta){
    ta.setAttribute('data-cminit', '1');
    var kind = ta.getAttribute('data-cm');
    if(kind === 'code'){ if(window.CodeMirror) boopMakeCM(ta, false); }
    else if(kind === 'composer'){ if(window.CodeMirror) boopMakeCM(ta, true); }
    else if(kind === 'edit'){
      ta.focus(); ta.setSelectionRange(ta.value.length, ta.value.length);
      ta.addEventListener('keydown', function(e){
        if((e.shiftKey||e.ctrlKey||e.metaKey) && e.key === 'Enter'){ e.preventDefault(); boopSave(ta.getAttribute('data-cid')); }
      });
    }
  });
  boopRenderAnsi(scope);
}
if(window.htmx) htmx.onLoad(boopInitEditors);
"""

In [ ]:
#| export
_THEME_JS = r"""
window._boopcms = window._boopcms || [];
var _HLJS = {
  dark:  'https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-dark.min.css',
  light: 'https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-light.min.css'
};
function boopApplyTheme(dark){
  // The data-theme flip below is what you actually SEE (DaisyUI restyles via CSS instantly) --
  // but JS blocks painting until this function returns, so re-theming every live CodeMirror
  // instance (slow with many code cells open) was delaying that paint by up to ~1.5s. Do the
  // visible part synchronously, defer the CodeMirror re-theme to right after so it can't block it.
  window._boopDark = dark;
  document.documentElement.setAttribute('data-theme', dark ? 'dark' : 'light');
  var hl = document.getElementById('hljs-theme');
  if(hl) hl.href = dark ? _HLJS.dark : _HLJS.light;
  try { localStorage.setItem('boopDark', dark ? '1' : '0'); } catch(e){}
  setTimeout(function(){
    // Re-theme on-screen editors first (so the visible part of the page finishes fast), then
    // let the browser paint, then catch up the rest -- rather than one big synchronous sweep.
    var vh = window.innerHeight;
    var onscreen = [], offscreen = [];
    window._boopcms.forEach(function(cm){
      var r = cm.getWrapperElement().getBoundingClientRect();
      (r.bottom >= 0 && r.top <= vh ? onscreen : offscreen).push(cm);
    });
    var retheme = function(cm){ cm.setOption('theme', dark ? 'material-darker' : 'default'); };
    onscreen.forEach(retheme);
    if(window._boopComposerCM) retheme(window._boopComposerCM);
    setTimeout(function(){ offscreen.forEach(retheme); }, 0);
  }, 0);
}
function boopThemeToggle(cb){ boopApplyTheme(cb.checked); }
document.addEventListener('DOMContentLoaded', function(){
  var saved = null; try { saved = localStorage.getItem('boopDark'); } catch(e){}
  var dark = (saved === null) ? true : (saved === '1');
  var cb = document.getElementById('theme-toggle');
  if(cb) cb.checked = dark;
  boopApplyTheme(dark);
});
"""

In [ ]:
#| export
def _tw_header():
    "Use the precompiled static Tailwind build (from `_build_tailwind()`) if it exists; else fall back to the slower CDN JIT compiler. `__file__` isn't defined when nbdev-test executes this notebook directly (vs. a real module import), so fall back to cwd -- the .exists() check below fails safely either way."
    pkg_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    compiled = pkg_dir/'static/tailwind.css'
    if compiled.exists(): return Link(rel='stylesheet', href='/tailwind.css')
    return Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4')

In [ ]:
#| export
_MARKDOWN_JS = r"""
import { marked } from "https://cdn.jsdelivr.net/npm/marked/lib/marked.esm.js";
import katex from "https://cdn.jsdelivr.net/npm/katex/dist/katex.mjs";

const renderMath = (tex, displayMode) => katex.renderToString(tex, {
    throwOnError: false, displayMode: displayMode, output: 'html', trust: true
});

const processLatexEnvironments = (content) => content.replace(/\\begin{(\w+)}([\s\S]*?)\\end{\1}/g, (match, env) => {
    return ['equation','align','gather','multline'].includes(env) ? `$$${match}$$` : match;
});

function boopHighlight(root){
    if (!window.hljs) return;
    root.querySelectorAll('pre code').forEach(c => hljs.highlightElement(c));
}

// proc_htmx (fasthtml-js) has no dedup of its own -- it refires on *every* htmx swap anywhere
// on the page. Guard with :not([data-md-done]) so already-rendered cells are never re-parsed
// (their innerHTML is no longer the raw markdown source, so re-parsing would corrupt them).
proc_htmx('.marked:not([data-md-done])', e => {
    let content = processLatexEnvironments(e.textContent);
    content = content.replace(/\$\$([\s\S]+?)\$\$/gm, (_, tex) => renderMath(tex.trim(), true));
    content = content.replace(/(?<!\w)\$([^\$\s](?:[^\$]*[^\$\s])?)\$(?!\w)/g, (_, tex) => renderMath(tex.trim(), false));
    e.innerHTML = marked.parse(content);
    e.setAttribute('data-md-done', '1');
    boopHighlight(e);
});
"""

In [ ]:
#| export
daisy_hdrs = [
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    _tw_header(),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.css'),
    Script(_MARKDOWN_JS, type='module'),
    Link(id='hljs-theme', rel='stylesheet',
         href='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-dark.min.css'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/highlight.min.js'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/languages/python.min.js'),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/codemirror@5/lib/codemirror.min.css'),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/codemirror@5/theme/material-darker.min.css'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/lib/codemirror.min.js'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/mode/python/python.min.js'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/addon/comment/comment.min.js'),
    Link(rel='icon', type='image/png', href='/logo.png'),
    Style(_MARKED_CSS),
    Script("import { AnsiUp } from 'https://cdn.jsdelivr.net/npm/ansi_up@6/ansi_up.js';"
           " window.AnsiUp = AnsiUp; if(window.boopRenderAnsi) boopRenderAnsi();", type='module'),
    Script(_THEME_JS),
    Script(_HOTKEYS_JS),
    Script(_EDIT_JS),
]

In [ ]:
#| export
app = FastHTML(hdrs=daisy_hdrs, htmlkw={'data-theme':'dark'})
rt  = app.route
p   = partial(HTMX, app=app, host=None, port=None)

## Code execution

A single IPython shell backs every code cell (like the lesson's `ex`), with errors
returned as text instead of raised.

In [ ]:
#| export
_shell = get_shell()
_shell.system = _shell.system_piped   # capture `!cmd` output into stdout

def run_code(src):
    "Execute `src` in the shared shell; return result, stdout, or an error string."
    res = _shell.run_cell(src)
    if res.error_in_exec is not None:
        e = res.error_in_exec
        return f'{type(e).__name__}: {e}'
    if res.result is not None: return res.result
    return (res.stdout or '').replace('\r\n', '\n')

## Cell + Notebook model

A `Cell` carries its type, source, optional output, and a `visible` flag (the eye
toggle — whether the LLM sees it). `Notebook` is the in-memory store of cells, the
composer's selected type, and the currently `selected` cell (for hotkeys).

In [ ]:
#| export
CTYPES = ('code','note','prompt','raw')  # types you can author; 'assistant' is generated

class Cell:
    def __init__(self, id, ctype, source, output=None, visible=True, model=None):
        self.id,self.ctype,self.source = id,ctype,source
        self.output,self.visible = output,visible
        self.model = model  # which LLM produced this (assistant cells only)
        self.ts = datetime.now().strftime('%I:%M:%S %p')


In [ ]:
#| export
class Notebook:
    def __init__(self):
        self.cells, self._nid, self.compose_type, self.selected = [], 0, 'code', None
        self.name = 'untitled'
        self.models, self.model = [], None  # available LLMs + the one currently selected in the top bar
        self.clipboard = []  # cell snapshots (plain dicts, not live Cells) for cut/copy/paste
        self.tools = []  # functions the LLM may call on Prompt-cell runs -- see add_tool()

    def insert_at(self, pos, ctype, source, output=None, visible=True, model=None):
        self._nid += 1
        c = Cell(self._nid, ctype, source, output, visible, model)
        self.cells.insert(pos, c)
        return c

    def add(self, ctype, source, output=None, visible=True, model=None):
        return self.insert_at(len(self.cells), ctype, source, output, visible, model)

    def index(self, id): return next((i for i,c in enumerate(self.cells) if c.id==id), None)
    def get(self, id):
        i = self.index(id)
        return self.cells[i] if i is not None else None
    def sel_index(self):
        return None if self.selected is None else self.index(self.selected)

    def pair_range(self, id):
        "Indices (start,end) spanning the Prompt+Assistant pair containing `id`, or just (i,i) for a lone cell."
        i = self.index(id)
        if i is None: return None
        c = self.cells[i]
        if c.ctype == 'prompt' and i+1 < len(self.cells) and self.cells[i+1].ctype == 'assistant':
            return (i, i+1)
        if c.ctype == 'assistant' and i-1 >= 0 and self.cells[i-1].ctype == 'prompt':
            return (i-1, i)
        return (i, i)

    def remove(self, id):
        "Delete the cell, or its whole Prompt+Assistant pair if it's part of one."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        del self.cells[lo:hi+1]

    def move(self, id, delta):
        "Move the cell (or its Prompt+Assistant pair) up/down as a block, swapping with whatever cell/pair is adjacent."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        if delta < 0:
            if lo == 0: return
            nlo, _ = self.pair_range(self.cells[lo-1].id)
            block, neighbor = self.cells[lo:hi+1], self.cells[nlo:lo]
            self.cells[nlo:hi+1] = block + neighbor
        elif delta > 0:
            if hi >= len(self.cells) - 1: return
            _, nhi = self.pair_range(self.cells[hi+1].id)
            block, neighbor = self.cells[lo:hi+1], self.cells[hi+1:nhi+1]
            self.cells[lo:nhi+1] = neighbor + block

    def _snapshot(self, lo, hi):
        return [{'ctype':c.ctype,'source':c.source,'output':c.output,'visible':c.visible,'model':c.model}
                for c in self.cells[lo:hi+1]]

    def copy_range(self, id):
        "Copy the cell (or its pair) into the clipboard, without removing it."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        self.clipboard = self._snapshot(lo, hi)

    def cut_range(self, id):
        "Copy the cell (or its pair) into the clipboard, then remove it."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        self.clipboard = self._snapshot(lo, hi)
        del self.cells[lo:hi+1]
        self.selected = self.cells[min(lo, len(self.cells)-1)].id if self.cells else None

    def paste_after(self, id):
        "Paste the clipboard as new cells (fresh ids) right after `id`'s pair, or at the end if id is None."
        if not self.clipboard: return []
        rng = self.pair_range(id) if id is not None else None
        pos = rng[1] + 1 if rng else len(self.cells)
        pasted = []
        for snap in self.clipboard:
            pasted.append(self.insert_at(pos, snap['ctype'], snap['source'], snap['output'], snap['visible'], snap['model']))
            pos += 1
        self.selected = pasted[0].id
        return pasted

    def reset(self):
        "Discard all cells and start over, as if boopiter had just launched fresh."
        self.cells.clear()
        self._nid, self.selected, self.name = 0, None, 'untitled'

nb = Notebook()
BROWSE_ROOT = Path.cwd()  # file browser is rooted here (wherever `boopiter` was launched from), like Jupyter


## LLM Interaction

The whole point of the visibility toggle: context is only the *visible* cells. The
stub proves the plumbing by reporting what it can see; swap `stub_reply` for a real
model call later.

In [ ]:
#| export 
def get_model_list(debug:bool=False): 
    "Get a list of supported (local) models"
    import httpx
    models = httpx.get("http://localhost:11434/api/tags").json()
    if debug: print("models = ",models)
    if not models.get('models'):
        raise RuntimeError("No local LLM models found -- is Ollama running, and do you have any models pulled?")
    return ['ollama/'+m['model'] for m in models['models']]

In [ ]:
#| eval: false
get_model_list()

In [ ]:
#| export
# lisette + local (Ollama) models: passing non-empty `tools=` combined with `tool_choice='none'`
# triggers a bug in litellm's MCP-handler codepath that returns a raw dict instead of a proper
# response object (AttributeError: 'dict' object has no attribute 'choices'). Same issue hit by
# SBrewer15/CellMate (https://github.com/SBrewer15/CellMate) -- this is their patch, adopted as-is:
# drop tool_schemas for just that one call whenever tool_choice=='none', then restore them after.
_orig_chat_call = Chat._call
@patch
def _call(self:Chat, msg=None, prefill=None, temp=None, think=None, search=None, stream=False,
          max_steps=2, step=1, final_prompt=None, tool_choice=None, max_tokens=None, **kwargs):
    "Internal method that always yields responses"
    _orig_tools = self.tool_schemas
    if tool_choice == 'none': self.tool_schemas, tool_choice = None, None
    try: yield from _orig_chat_call(self, msg, prefill, temp, think, search, stream, max_steps, step, final_prompt, tool_choice, max_tokens, **kwargs)
    finally: self.tool_schemas = _orig_tools

def prompt_llm(context:str, model:str='ollama/qwen2.5-coder:latest', tools=[]): 
    "Send a prompt to the LLM and return its response. TODO: can we stream the response rather than wait for as a final big chunk?"
    # _skip_mcp_handler avoids litellm's MCP-proxy import chain (needs fastapi/orjson) that we don't use.
    # Drop it (and re-add fastapi/orjson to pyproject.toml) if/when we actually want MCP tool support.
    chat = Chat(model, tools=tools, callkw={'_skip_mcp_handler': True}) # FYI: this makes a fresh stateless context each time. is that what we want?
    response = chat(context)
    return contents(response).content

def add_tool(fn):
    "Register `fn` as a tool the LLM can call on future Prompt-cell runs. Also usable as a decorator: `@add_tool`."
    if not callable(fn):
        raise TypeError(f'add_tool() expects a callable, got {fn!r}')
    if fn not in nb.tools:
        nb.tools.append(fn)
    return fn

_shell.push({'nb': nb, 'add_tool': add_tool})  # so code cells can call add_tool(...)/inspect nb directly, no import needed


In [ ]:
#| eval: false
c = prompt_llm("Today is July 18. Who's one famous person with this birthday?") 
print(str(c))
prompt_llm("Tell me the previous question I asked you, from the previous prompt. I want to see if you retain state between calls")

As of today, July 18, there isn't a widely recognized or notable public figure who shares this exact birthdate. Birthdays for celebrities and historical figures are often celebrated around the world, so it's possible that someone you're thinking of might have had their birthday on a different date but is still famous.

If you're looking for a specific type of person (e.g., an athlete, artist, scientist), please provide more details!


In [ ]:
#| export
def llm_context(nb, cur_id=None):
    "Exactly what a real model would receive: the visible cells up through `cur_id` (default: all), in order."
    cutoff = len(nb.cells)
    if cur_id is not None:
        i = nb.index(cur_id)
        if i is not None: cutoff = i + 1
    return '\n'.join(f'[{c.ctype}] {c.source}' for c in nb.cells[:cutoff] if c.visible)

In [ ]:
#| export
def stub_reply(nb, prompt):
    "fake/placeholder reply in case llms aren't available (can still test gui)"
    n = sum(c.visible for c in nb.cells)
        # TODO: use the prompt_llm routine instead to get real llm interaction
    return (f'(stub) I can see {n} visible cell(s). You said: '
            f'"{prompt.strip()}". Wire a real model into stub_reply() later.')

In [ ]:
#| export
_PREFERRED_MODEL_SUBSTR = 'qwen2.5-coder'  # used if present, regardless of exact tag/version

def ensure_models():
    "Populate nb.models/nb.model once at startup, tolerating an unreachable local LLM server."
    try:
        nb.models = get_model_list()
    except Exception:
        nb.models = []
    preferred = next((m for m in nb.models if _PREFERRED_MODEL_SUBSTR in m), None)
    nb.model = preferred or (nb.models[0] if nb.models else None)

try:
    @app.on_event('startup')
    def _load_models():
        ensure_models()
except: 
    print("WARNING: Can't test this cell in notebook, no app")

### Tool Use

Example tool from lisette docs:

In [ ]:

def add_numbers(
    a: int,  # First number to add
    b: int   # Second number to add  
) -> int:
    "Add two numbers together"
    return a + b

In [ ]:
#| eval: false
res = prompt_llm("What's 47 + 23? Use the tool.", tools=[add_numbers])
print(res)

I apologize for any confusion, but it appears there was a misunderstanding in our interaction. As an AI language model created by Alibaba Cloud, I don't have "tool calls" or a specific capability to perform calculations like addition. However, I can certainly help you with such simple arithmetic problems directly through text.

To answer your question: 47 + 23 equals 70.

If you need further assistance with any other questions or tasks, feel free to ask!


## Rendering

Each type gets a colored left border (matching the SolveIt screenshot: raw=yellow,
code=blue, note=green, prompt/assistant=red). The selected cell gets a ring;
hidden-from-LLM cells are dimmed. Note cells render as markdown via the `.marked`
class (KaTeX, images, HTML).

In [ ]:
#| export
BORDER = {'raw':'border-warning', 'code':'border-info', 'note':'border-success',
          'prompt':'border-error', 'assistant':'border-error'}

# heroicons (outline, 1.5 stroke) -- https://heroicons.com
ICONS = {
    'copy':          ('M8.25 9V5.25A2.25 2.25 0 0 1 10.5 3h6a2.25 2.25 0 0 1 2.25 2.25v13.5A2.25 2.25 0 0 1 16.5 21h-6a2.25 2.25 0 0 1-2.25-2.25V15m-3 0-3-3m0 0 3-3m-3 3H15',),
    'eye':           ('M2.036 12.322a1.012 1.012 0 0 1 0-.639C3.423 7.51 7.36 4.5 12 4.5c4.638 0 8.573 3.007 9.963 7.178.07.207.07.431 0 .639C20.577 16.49 16.64 19.5 12 19.5c-4.638 0-8.573-3.007-9.963-7.178Z',
                       'M15 12a3 3 0 1 1-6 0 3 3 0 0 1 6 0Z'),
    'eye-slash':     ('M3.98 8.223A10.477 10.477 0 0 0 1.934 12C3.226 16.338 7.244 19.5 12 19.5c.993 0 1.953-.138 2.863-.395M6.228 6.228A10.451 10.451 0 0 1 12 4.5c4.756 0 8.773 3.162 10.065 7.498a10.522 10.522 0 0 1-4.293 5.774M6.228 6.228 3 3m3.228 3.228 3.65 3.65m7.894 7.894L21 21m-3.228-3.228-3.65-3.65m0 0a3 3 0 1 0-4.243-4.243m4.242 4.242L9.88 9.88',),
    'trash':         ('m14.74 9-.346 9m-4.788 0L9.26 9m9.968-3.21c.342.052.682.107 1.022.166m-1.022-.165L18.16 19.673a2.25 2.25 0 0 1-2.244 2.077H8.084a2.25 2.25 0 0 1-2.244-2.077L4.772 5.79m14.456 0a48.108 48.108 0 0 0-3.478-.397m-12 .562c.34-.059.68-.114 1.022-.165m0 0a48.11 48.11 0 0 1 3.478-.397m7.5 0v-.916c0-1.18-.91-2.164-2.09-2.201a51.964 51.964 0 0 0-3.32 0c-1.18.037-2.09 1.022-2.09 2.201v.916m7.5 0a48.667 48.667 0 0 0-7.5 0',),
    'arrow-up':      ('M4.5 10.5 12 3m0 0 7.5 7.5M12 3v18',),
    'arrow-down':    ('M19.5 13.5 12 21m0 0-7.5-7.5M12 21V3',),
    'arrow-path':    ('M16.023 9.348h4.992v-.001M2.985 19.644v-4.992m0 0h4.992m-4.993 0 3.181 3.183a8.25 8.25 0 0 0 13.803-3.7M4.031 9.865a8.25 8.25 0 0 1 13.803-3.7l3.181 3.182m0-4.991v4.99',),
    'x-circle':      ('m9.75 9.75 4.5 4.5m0-4.5-4.5 4.5M21 12a9 9 0 1 1-18 0 9 9 0 0 1 18 0Z',),
    'play':          ('M5.25 5.653c0-.856.917-1.398 1.667-.986l11.54 6.347a1.125 1.125 0 0 1 0 1.972l-11.54 6.347a1.125 1.125 0 0 1-1.667-.986V5.653Z',),
    'play-circle':   ('M21 12a9 9 0 1 1-18 0 9 9 0 0 1 18 0Z',
                       'M15.91 11.672a.375.375 0 0 1 0 .656l-5.603 3.113a.375.375 0 0 1-.557-.328V8.887c0-.286.307-.466.557-.327l5.603 3.112Z'),
    'bookmark':      ('M17.593 3.322c1.1.128 1.907 1.077 1.907 2.185V21L12 17.25 4.5 21V5.507c0-1.108.806-2.057 1.907-2.185a48.507 48.507 0 0 1 11.186 0Z',),
    'bars-3':        ('M3.75 6.75h16.5M3.75 12h16.5m-16.5 5.25h16.5',),
    'folder':        ('M2.25 12.75V12A2.25 2.25 0 0 1 4.5 9.75h15A2.25 2.25 0 0 1 21.75 12v.75m-8.69-6.44-2.12-2.12a1.5 1.5 0 0 0-1.061-.44H4.5A2.25 2.25 0 0 0 2.25 6v12a2.25 2.25 0 0 0 2.25 2.25h15A2.25 2.25 0 0 0 21.75 18V9a2.25 2.25 0 0 0-2.25-2.25h-5.379a1.5 1.5 0 0 1-1.06-.44Z',),
    'document-text': ('M19.5 14.25v-2.625a3.375 3.375 0 0 0-3.375-3.375h-1.5A1.125 1.125 0 0 1 13.5 7.125v-1.5a3.375 3.375 0 0 0-3.375-3.375H8.25m0 12.75h7.5m-7.5 3H12M10.5 2.25H5.625c-.621 0-1.125.504-1.125 1.125v17.25c0 .621.504 1.125 1.125 1.125h12.75c.621 0 1.125-.504 1.125-1.125V11.25a9 9 0 0 0-9-9Z',),
}

def Icon(name, cls='size-4'):
    "A heroicons outline SVG, inlined so `stroke='currentColor'` matches the button's text color."
    return fh.Svg(*[SvgPath(stroke_linecap='round', stroke_linejoin='round', d=d) for d in ICONS[name]],
                  xmlns='http://www.w3.org/2000/svg', fill='none', viewbox='0 0 24 24',
                  stroke_width='1.5', stroke='currentColor', cls=cls)

def IconBtn(name, title, **kw):
    return fh.Button(Icon(name), cls='btn btn-sm btn-ghost', title=title, **kw)

# nbdev's '#| export' pragma, as a leading line in a code cell's source. We hide the literal
# text from the user and represent it with a toggleable bookmark icon instead.
def _has_export(source):
    return source.split('\n', 1)[0].strip().replace(' ', '') == '#|export'

def _strip_export(source):
    'The source with any leading #| export pragma line removed -- what the user sees/edits.'
    if not _has_export(source): return source
    rest = source.split('\n', 1)
    return rest[1] if len(rest) > 1 else ''

def _set_export(source, flag):
    "Add or remove the leading '#| export' pragma line so the source matches `flag`."
    stripped = _strip_export(source)
    return f'#| export\n{stripped}' if flag else stripped


In [ ]:
#| export
def cell_toolbar(c):
    tgt = '#notebook'
    copy_btn = fh.Button(Icon('copy'), id=f'copy-{c.id}', title='Copy to clipboard', type='button',
                         cls='btn btn-sm btn-ghost', data_src=c.source,
                         onclick=f"boopCopy({c.id}, '{c.ctype}')")
    btns = [copy_btn]
    if c.ctype == 'code':
        exported = _has_export(c.source)
        btns.append(fh.Button(Icon('bookmark'), title='Exported (#| export)' if exported else 'Not exported',
                              type='button', hx_post=toggle_export.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML',
                              cls='btn btn-sm btn-ghost' + (' text-error' if exported else '')))
    btns.append(IconBtn('eye' if c.visible else 'eye-slash',
                    'Hide from LLM' if c.visible else 'Show to LLM',
                    hx_post=toggle_vis.to(id=c.id), hx_target=tgt))
    if c.ctype == 'code':
        btns.append(IconBtn('play', 'Run', onclick=f'boopSave({c.id})'))
    elif c.ctype == 'prompt':
        btns.append(IconBtn('play', 'Run', hx_post=run_cell.to(id=c.id), hx_target=tgt))
    btns += [
        IconBtn('arrow-up', 'Move up',   hx_post=move_cell.to(id=c.id, delta=-1), hx_target=tgt),
        IconBtn('arrow-down', 'Move down', hx_post=move_cell.to(id=c.id, delta=1),  hx_target=tgt),
        IconBtn('trash', 'Delete', hx_post=del_cell.to(id=c.id), hx_target=tgt),
    ]
    return Div(*btns, cls='flex gap-1 ml-auto')


In [ ]:
#| export
def type_dropdown(c):
    "Click the cell-type word to switch it (code/note/prompt/raw). Scoped to just this cell."
    if c.ctype == 'assistant':
        return Span('Assistant', cls='font-semibold text-sm')
    opts = [Li(fh.A(t.capitalize(), hx_post=set_ctype.to(id=c.id, t=t),
                    hx_target=f'#cell-{c.id}', hx_swap='outerHTML'))
            for t in CTYPES if t != c.ctype]
    return Div(
        Div(c.ctype.capitalize(), tabindex='0', role='button',
            cls='font-semibold text-sm cursor-pointer'),
        Ul(*opts, tabindex='0', cls='dropdown-content menu bg-base-200 rounded-box z-10 w-28 p-1 shadow'),
        cls='dropdown dropdown-bottom')

def cell_header(c):
    rest = f': {c.id}' + (f' ({c.ts})' if c.ctype in ('code','assistant') else '')
    if c.ctype == 'assistant' and c.model: rest += f' \xb7 {c.model}'
    return Div(type_dropdown(c),
               Span(rest, cls='font-semibold text-sm cursor-pointer flex-1',
                    hx_post=select.to(id=c.id), hx_target='#notebook'),
               cell_toolbar(c), cls='flex items-center gap-2 mb-1')

def cell_body(c):
    "Note/prompt/assistant render as markdown; raw is bare text. Code cells never reach here -- render_cell() routes them to code_editor()."
    if c.ctype in ('note', 'prompt', 'assistant'):  # markdown + KaTeX + HTML + images + highlighted code fences
        return Div(c.source, cls='marked prose max-w-none')
    return Pre(c.source, cls='font-mono text-sm whitespace-pre-wrap')

def _cell_outer(c, *content):
    dim    = '' if c.visible else 'opacity-40'
    indent = 'ml-8' if c.ctype == 'assistant' else ''
    ring   = 'ring-2 ring-primary ring-offset-2 ring-offset-base-100 rounded' if c.id == nb.selected else ''
    return Div(*content, id=f'cell-{c.id}',
               cls=f'border-l-4 {BORDER[c.ctype]} pl-3 py-2 my-2 {dim} {indent} {ring}')

def code_editor(c):
    "Code cells are always a live CodeMirror editor; Shift/Ctrl/Cmd+Enter or the play button runs. The '#| export' pragma (if any) is hidden -- see the bookmark toggle in cell_toolbar."
    src = _strip_export(c.source)
    ta = Textarea(src, name='source', id=f'ta-{c.id}',
                  rows=str(max(2, src.count(chr(10)) + 1)),
                  cls='textarea textarea-bordered w-full font-mono',
                  data_cm='code', data_cid=str(c.id))
    parts = [Form(ta, hx_post=save_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')]
    if c.output not in (None, ''):
        parts.append(Pre(str(c.output), cls='ansi-out text-sm mt-1 whitespace-pre overflow-x-auto'))
    return Div(*parts)

def render_cell(c):
    "Code cells are always editors; note/raw/assistant render and open an editor on click."
    if c.ctype == 'code':
        return _cell_outer(c, cell_header(c), code_editor(c))
    body = cell_body(c)
    body = Div(body, cls='cursor-text',
                   hx_get=edit_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')
    return _cell_outer(c, cell_header(c), body)

def render_cell_edit(c):
    "Inline editor. Code cells use CodeMirror (Python highlight, no wrap); notes/raw use a textarea. Shift/Ctrl/Cmd+Enter saves."
    ta = Textarea(c.source, name='source', id=f'ta-{c.id}',
                  rows=str(max(3, c.source.count(chr(10)) + 2)),
                  cls='textarea textarea-bordered w-full font-mono',
                  data_cm='edit', data_cid=str(c.id))
    buttons = Div(Button('Save', type='button', cls='btn btn-primary btn-xs',
                         onclick=f'boopSave({c.id})'),
                  Button('Cancel', type='button', cls='btn btn-ghost btn-xs',
                         hx_get=view_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'),
                  cls='flex gap-2 justify-end mt-1')
    form = Form(ta, buttons, hx_post=save_cell.to(id=c.id),
                hx_target=f'#cell-{c.id}', hx_swap='outerHTML')
    return _cell_outer(c, cell_header(c), form)

def render_nb():
    return Div(*[render_cell(c) for c in nb.cells], id='notebook', cls='flex flex-col')

## Composer

The bottom bar: type tabs, a textarea, Submit. Picking a tab sets the type
server-side; Submit creates the cell (running it, if code; spawning an Assistant
reply, if prompt).

In [ ]:
#| export
def composer(draft='', oob=False):
    tabs = [fh.A(t.capitalize(),
                 cls=f'tab {"tab-active" if nb.compose_type==t else ""}',
                 hx_post=set_type.to(t=t), hx_target='#composer', hx_swap='outerHTML')
            for t in CTYPES]
    ta_kw = {'data_cm':'composer'} if nb.compose_type=='code' else {}
    div_kw = {'hx_swap_oob':'true'} if oob else {}
    return Div(
        Div(*tabs, cls='tabs tabs-boxed'),
        Form(Textarea(draft, placeholder=f'{nb.compose_type} cell…', name='source',
                      id='compose-input', rows='3',
                      onkeydown="if((event.shiftKey||event.ctrlKey||event.metaKey)&&event.key==='Enter')"
                               "{event.preventDefault();this.form.requestSubmit();}",
                      cls='textarea textarea-bordered w-full font-mono', **ta_kw),
             Div(Button('Boop', type='button', onclick='boopComposerSubmit()', cls='btn btn-primary'), cls='flex justify-end mt-2'),
             hx_post=submit_cell, hx_target='#notebook', hx_swap='beforeend'),
        id='composer', cls='border-t border-base-300 pt-3 mt-4', **div_kw)

def render_app(draft=''):
    return Div(render_nb(), composer(draft), id='app')

## Routes

Composer/toolbar routes plus the command-mode routes driven by hotkeys
(`select`, `select_delta`, `insert`, `del_selected`, `settype_selected`).

In [ ]:
#| export
# ---- top menu / control bar ----
def theme_swap():
    "DaisyUI sun/moon swap; drives boopApplyTheme (default dark)."
    return NotStr('<label class="swap swap-rotate btn btn-ghost btn-circle btn-sm" title="Toggle light/dark">'
      '<input type="checkbox" id="theme-toggle" onchange="boopThemeToggle(this)" checked />'
      '<svg class="swap-off h-5 w-5 fill-current" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
      '<path d="M5.64,17l-.71.71a1,1,0,0,0,0,1.41,1,1,0,0,0,1.41,0l.71-.71A1,1,0,0,0,5.64,17ZM5,12a1,1,0,0,0-1-1H3a1,1,0,0,0,0,2H4A1,1,0,0,0,5,12Zm7-7a1,1,0,0,0,1-1V3a1,1,0,0,0-2,0V4A1,1,0,0,0,12,5ZM5.64,7.05a1,1,0,0,0,.7.29,1,1,0,0,0,.71-.29,1,1,0,0,0,0-1.41l-.71-.71A1,1,0,0,0,4.93,6.34Zm12,.29a1,1,0,0,0,.7-.29l.71-.71a1,1,0,1,0-1.41-1.41L17,5.64a1,1,0,0,0,0,1.41A1,1,0,0,0,17.66,7.34ZM21,11H20a1,1,0,0,0,0,2h1a1,1,0,0,0,0-2Zm-9,8a1,1,0,0,0-1,1v1a1,1,0,0,0,2,0V20A1,1,0,0,0,12,19ZM18.36,17A1,1,0,0,0,17,18.36l.71.71a1,1,0,0,0,1.41,0,1,1,0,0,0,0-1.41ZM12,6.5A5.5,5.5,0,1,0,17.5,12,5.51,5.51,0,0,0,12,6.5Zm0,9A3.5,3.5,0,1,1,15.5,12,3.5,3.5,0,0,1,12,15.5Z"/></svg>'
      '<svg class="swap-on h-5 w-5 fill-current" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
      '<path d="M21.64,13a1,1,0,0,0-1.05-.14,8.05,8.05,0,0,1-3.37.73A8.15,8.15,0,0,1,9.08,5.49a8.59,8.59,0,0,1,.25-2A1,1,0,0,0,8,2.36,10.14,10.14,0,1,0,22,14.05,1,1,0,0,0,21.64,13Zm-9.5,6.69A8.14,8.14,0,0,1,7.08,5.22v.27A10.15,10.15,0,0,0,17.22,15.63a9.79,9.79,0,0,0,2.1-.22A8.11,8.11,0,0,1,12.14,19.73Z"/></svg></label>')

_BOOP2NB = {'code':'code', 'note':'markdown', 'prompt':'markdown', 'raw':'raw', 'assistant':'markdown'}

def save_notebook(path=None):
    "Serialize `nb.cells` to a real Jupyter notebook file (`{nb.name}.ipynb` in the cwd, by default)."
    path = Path(path) if path else Path.cwd()/f'{nb.name}.ipynb'
    doc = _nbf.v4.new_notebook()
    for c in nb.cells:
        meta = {'boopiter': {'ctype': c.ctype, 'visible': c.visible}}
        kind = _BOOP2NB.get(c.ctype, 'raw')
        if kind == 'code':
            outputs = [_nbf.v4.new_output('stream', name='stdout', text=str(c.output))] if c.output not in (None, '') else []
            cell = _nbf.v4.new_code_cell(c.source, outputs=outputs, metadata=meta)
        elif kind == 'markdown':
            cell = _nbf.v4.new_markdown_cell(c.source, metadata=meta)
        else:
            cell = _nbf.v4.new_raw_cell(c.source, metadata=meta)
        doc.cells.append(cell)
    _nbf.write(doc, str(path))
    return path

_NB_FALLBACK = {'code':'code', 'markdown':'note', 'raw':'raw'}

def load_notebook(path):
    "Load a Jupyter notebook file into `nb`, replacing its current contents. Inverse of save_notebook()."
    path = Path(path)
    doc = _nbf.read(str(path), as_version=4)
    nb.cells.clear()
    nb._nid = 0
    nb.selected = None
    for cell in doc.cells:
        meta = cell.get('metadata', {}).get('boopiter', {})
        ctype = meta.get('ctype')
        if ctype not in CTYPES + ('assistant',):
            ctype = _NB_FALLBACK.get(cell.cell_type, 'raw')  # plain (non-boopiter) notebook
        output = None
        if ctype == 'code':
            texts = [o.get('text','') for o in cell.get('outputs', []) if o.get('output_type') == 'stream']
            output = ''.join(texts) or None
        nb.add(ctype, cell.source, output=output, visible=meta.get('visible', True))
    nb.name = str(path.with_suffix(''))  # keep the directory, only strip .ipynb
    return nb

def fname_display():
    return Span(nb.name, id='fname', title='Click to rename',
                cls='cursor-pointer font-mono opacity-80 hover:opacity-100',
                hx_get=rename_form, hx_target='#fname', hx_swap='outerHTML')

@rt
def rename_form():
    return Form(Input(value=nb.name, name='name',
                      cls='input input-sm input-bordered font-mono',
                      onkeydown="if(event.key===\'Escape\'){this.form.requestSubmit();}"),
                Script("var i=document.querySelector(\'#fname input\'); if(i){i.focus();i.select();}"),
                id='fname', hx_post=rename, hx_target='#fname', hx_swap='outerHTML')

def model_dropdown():
    "Select which LLM answers Prompt cells; selection lives on `nb.model`."
    if not nb.models:
        return Span('no models', cls='text-xs opacity-50', title='No local LLMs found (is Ollama running?)')
    opts = [fh.Option(m, value=m, selected=(m == nb.model)) for m in nb.models]
    return fh.Select(*opts, name='model', cls='select select-sm select-bordered',
                      hx_post=set_model, hx_trigger='change', hx_swap='none')

@rt
def set_model(model:str):
    if model in nb.models and model != nb.model:
        old, nb.model = nb.model, model
        if old and old.startswith('ollama/'):
            import subprocess
            try: subprocess.run(['ollama', 'stop', old.removeprefix('ollama/')], capture_output=True, timeout=5)
            except Exception: pass
    return ''

def file_menu():
    "Hamburger dropdown: New / Open (file browser) / Save / Download / Restart Server."
    items = [
        Li(fh.A('New', href=new_notebook.to(),
                onclick="return confirm('Discard the current notebook and start a new one?')")),
        Li(fh.A('Open', onclick="document.getElementById('file-modal').showModal()",
                hx_get=browse.to(), hx_target='#file-browser-body', hx_swap='innerHTML')),
        Li(fh.A('Save', href='javascript:void(0)', onclick='boopSaveNotebook()')),
        Li(fh.A('Download', href=download.to())),
        Li(fh.A('Restart Server', href='javascript:void(0)', onclick='boopRestartServer()')),
    ]
    return Div(
        Div(Icon('bars-3'), tabindex='0', role='button', cls='btn btn-ghost btn-circle btn-sm'),
        Ul(*items, tabindex='0', cls='dropdown-content menu bg-base-200 rounded-box z-10 w-40 p-1 shadow'),
        cls='dropdown dropdown-bottom')

def file_browser_modal():
    return Dialog(
        Div(
            Div('Open notebook', cls='font-semibold mb-2'),
            Div(id='file-browser-body'),
            Div(Form(fh.Button('Close', cls='btn btn-sm'), method='dialog'), cls='modal-action'),
            cls='modal-box'),
        id='file-modal', cls='modal')

def top_bar():
    brand = Div(file_menu(), Img(src='/logo.png', cls='h-8 w-8 rounded-full'),
                Span('boopiter', cls='font-bold text-lg'),
                Span('/', cls='opacity-40'), fname_display(),
                cls='flex items-center gap-2')
    ctrls = Div(
        model_dropdown(),
        fh.Button(Icon('x-circle'), title='Interrupt kernel', cls='btn btn-ghost btn-circle btn-sm',
                  hx_post=interrupt_kernel, hx_swap='none'),
        fh.Button(Icon('arrow-path'), title='Restart kernel', cls='btn btn-ghost btn-circle btn-sm',
                  hx_post=restart_kernel, hx_swap='none'),
        fh.Button(Icon('play-circle'), title='Run all code cells', cls='btn btn-ghost btn-circle btn-sm',
                  hx_post=run_all, hx_target='#notebook', hx_swap='outerHTML'),
        theme_swap(), cls='flex items-center gap-1')
    return Div(brand, ctrls, file_browser_modal(), cls='navbar bg-base-200 shadow px-4 flex justify-between shrink-0')

@rt('/_boopiter_ping')
def boopiter_ping():
    "Identity check so `boopiter launch` can tell a live boopiter instance apart from something else on the port."
    return 'boopiter'

@rt('/logo.png')
def logo_png():
    return FileResponse(Path(__file__).parent.parent/'images/logo.png')

@rt('/tailwind.css')
def tailwind_css():
    return FileResponse(Path(__file__).parent/'static/tailwind.css')

@rt
def index():
    return (Title('boopiter'),
            Div(top_bar(),
                Div(Div(render_app(), cls='max-w-3xl mx-auto p-4'),
                    cls='flex-1 overflow-y-auto'),
                Div(id='save-toast', cls='toast toast-top toast-end z-50'),
                cls='h-screen flex flex-col'))

@rt
def _toast(msg, ok=True):
    "A little 'Saved' (or error) notice, out-of-band-swapped into #save-toast, that clears itself after ~1.8s."
    return Div(
        Div(msg, cls=f"alert {'alert-success' if ok else 'alert-error'} shadow-lg text-sm py-2 px-4"),
        Script("setTimeout(function(){ var t=document.getElementById('save-toast'); if(t) t.innerHTML=''; }, 1800)"),
        id='save-toast', cls='toast toast-top toast-end z-50', hx_swap_oob='true')

@rt
def save_now():
    try:
        p = save_notebook()
        return _toast(f'Saved {p.name}')
    except Exception as e:
        return _toast(f'Save failed: {e}', ok=False)

def _safe_dir(path):
    "Resolve `path` (relative to BROWSE_ROOT) and clamp it back to BROWSE_ROOT if it tries to escape (e.g. via '..')."
    cur = (BROWSE_ROOT/(path or '')).resolve()
    if cur != BROWSE_ROOT and BROWSE_ROOT not in cur.parents: cur = BROWSE_ROOT
    if not cur.is_dir(): cur = BROWSE_ROOT
    return cur

@rt
def browse(path:str=None):
    "Jupyter-tree-style directory listing for the file-browser modal, rooted at BROWSE_ROOT."
    cur = _safe_dir(path)
    rel = cur.relative_to(BROWSE_ROOT)
    entries = sorted(cur.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    rows = []
    if cur != BROWSE_ROOT:
        up = '' if rel.parent == Path('.') else str(rel.parent)
        rows.append(fh.A(Icon('folder'), ' ..', hx_get=browse.to(path=up),
                          hx_target='#file-browser-body', cls='flex items-center gap-2 py-1'))
    for p in entries:
        if p.name.startswith('.'): continue
        relp = str(p.relative_to(BROWSE_ROOT))
        if p.is_dir():
            rows.append(fh.A(Icon('folder'), ' ' + p.name, hx_get=browse.to(path=relp),
                              hx_target='#file-browser-body', cls='flex items-center gap-2 py-1'))
        elif p.suffix == '.ipynb':
            rows.append(fh.A(Icon('document-text'), ' ' + p.name, href=open_file.to(path=relp),
                              cls='flex items-center gap-2 py-1'))
        else:
            rows.append(Div(Icon('document-text'), Span(p.name), cls='flex items-center gap-2 py-1 opacity-40'))
    return Div(Div('/' if str(rel) == '.' else f'/{rel}', cls='font-mono text-xs opacity-60 mb-2'),
               *rows, id='file-browser-body')

@rt
def open_file(path:str):
    "Open a notebook found by the file browser; full navigation so the whole page reflects the new notebook."
    target = _safe_dir(Path(path).parent)/Path(path).name
    try: load_notebook(target.relative_to(BROWSE_ROOT))  # relative, to match the CLI's nb.name style
    except Exception: pass
    return index()

@rt
def new_notebook():
    nb.reset()
    return index()

@rt
def restart_server():
    "nbdev-export the notebooks as a real (blocking) subprocess -- so we get a genuine exit code instead of guessing a delay -- then clear __pycache__ (WSL/Windows-mounted filesystems can have coarse mtime resolution, which can trick Python into serving stale cached bytecode right after a fast save-then-restart) and exec a fresh copy of this process (same PID, same port). Aborts (leaves the current process running) if export fails. Does NOT save the notebook first; Save manually beforehand if you want to keep unsaved edits."
    def _do_restart():
        time.sleep(0.3)  # let the HTTP response reach the browser before we touch anything disruptive
        exe = Path(sys.executable).parent/'nbdev-export'
        exe = str(exe) if exe.exists() else 'nbdev-export'
        result = subprocess.run([exe], capture_output=True, text=True)
        if result.returncode != 0:
            print(f'nbdev-export failed ({result.returncode}), aborting restart:\n{result.stderr}', file=sys.stderr, flush=True)
            return
        cache_dir = Path(__file__).parent/'__pycache__'
        if cache_dir.exists(): shutil.rmtree(cache_dir, ignore_errors=True)
        port = os.environ.get('BOOPITER_PORT', '8000')
        args = [sys.executable, sys.argv[0]]
        if nb.name != 'untitled': args.append(f'{nb.name}.ipynb')
        args += ['--port', port]
        os.execv(sys.executable, args)
    threading.Thread(target=_do_restart, daemon=True).start()
    return ''

@rt
def download():
    p = save_notebook()
    return FileResponse(str(p), filename=p.name)

@rt
def rename(name:str):
    "Rename and persist the notebook to `{new_name}.ipynb` in the server's cwd."
    nb.name = name.strip() or nb.name
    save_notebook()
    return fname_display()

@rt
def restart_kernel():
    _shell.reset()          # clear kernel namespace
    return ''

@rt
def run_all():
    "Run every Code cell, top to bottom, in place (Prompt/Note/Raw/Assistant cells are left untouched)."
    for c in nb.cells:
        if c.ctype == 'code': c.output = run_code(c.source)
    return render_nb()

@rt
def interrupt_kernel():
    return ''               # placeholder: true interrupt needs threaded/async execution

@rt
def set_type(t:str):
    if t in CTYPES: nb.compose_type = t
    return composer()

def run_prompt_cell(id):
    "(Re)send `id`'s prompt -- plus everything visible above it -- to the LLM; create or refresh the paired Assistant cell. Returns that cell."
    c = nb.get(id)
    if not c or c.ctype != 'prompt': return None
    if nb.model:
        try: reply = prompt_llm(llm_context(nb, c.id), model=nb.model, tools=nb.tools)
        except Exception as e: reply = f'(error calling {nb.model}: {e})'
    else:
        reply = stub_reply(nb, c.source)
    i = nb.index(c.id)
    nxt = nb.cells[i+1] if i+1 < len(nb.cells) else None
    if nxt is not None and nxt.ctype == 'assistant':
        nxt.source, nxt.model = reply, nb.model
        nxt.ts = datetime.now().strftime('%I:%M:%S %p')
    else:
        nxt = nb.insert_at(i+1, 'assistant', reply, model=nb.model)
    return nxt

def pending_cell(prompt_id):
    "Placeholder shown right after a prompt is submitted; hx-trigger=load immediately fires the real (slow) LLM call and swaps itself out for the real Assistant cell once it replies."
    return Div('Assistant: Tricky...', id=f'pending-{prompt_id}',
               cls='border-l-4 border-error pl-3 py-2 my-2 ml-8 opacity-60 italic',
               hx_post=run_prompt_pending.to(id=prompt_id), hx_target=f'#pending-{prompt_id}',
               hx_swap='outerHTML', hx_trigger='load')

@rt
def run_prompt_pending(id:int):
    c2 = run_prompt_cell(id)
    return render_cell(c2) if c2 else ''

def add_cell(t, source):
    "Create a cell of type `t`: run it if code, just create it if prompt (its Assistant reply is added asynchronously via pending_cell). Returns the new cell(s)."
    if t == 'code':
        return [nb.add('code', source, output=run_code(source))]
    elif t == 'prompt':
        return [nb.add('prompt', source)]
    else:
        return [nb.add(t, source)]

@rt
def submit_cell(source:str):
    "Append only the new cell(s) to #notebook and reset the composer out-of-band, so untouched cells' editors are never re-created. A just-submitted Prompt gets a 'Thinking...' placeholder that fetches its own reply."
    new = add_cell(nb.compose_type, source) if source.strip() else []
    pending = [pending_cell(new[-1].id)] if new and nb.compose_type == 'prompt' else []
    return *[render_cell(c) for c in new], *pending, composer(oob=True)

@rt
def split(source:str, pos:int):
    "Split the composer at the caret: head becomes a cell, tail stays in the composer."
    head, tail = source[:pos], source[pos:]
    new = add_cell(nb.compose_type, head) if head.strip() else []
    pending = [pending_cell(new[-1].id)] if new and nb.compose_type == 'prompt' else []
    return *[render_cell(c) for c in new], *pending, composer(draft=tail, oob=True)

@rt
def run_cell(id:int):
    c = nb.get(id)
    if c and c.ctype == 'code': c.output = run_code(c.source)
    elif c and c.ctype == 'prompt': run_prompt_cell(id)
    return render_nb()

@rt
def toggle_vis(id:int):
    c = nb.get(id)
    if c: c.visible = not c.visible
    return render_nb()

@rt
def toggle_export(id:int):
    c = nb.get(id)
    if c and c.ctype == 'code':
        c.source = _set_export(c.source, not _has_export(c.source))
    return render_cell(c) if c else render_nb()

@rt
def del_cell(id:int):
    nb.remove(id)
    return render_nb()

@rt
def move_cell(id:int, delta:int):
    nb.move(id, delta)
    return render_nb()


NameError: name 'rt' is not defined

In [ ]:
#| export
# --- command-mode (hotkey) routes ---
@rt
def select(id:int):
    nb.selected = id
    return render_nb()

@rt
def select_delta(delta:int):
    if nb.cells:
        i = nb.sel_index()
        i = (0 if delta > 0 else len(nb.cells)-1) if i is None else min(max(i+delta, 0), len(nb.cells)-1)
        nb.selected = nb.cells[i].id
    return render_nb()

@rt
def insert(where:str):
    i = nb.sel_index()
    pos = len(nb.cells) if i is None else (i if where == 'above' else i+1)
    nb.selected = nb.insert_at(pos, nb.compose_type, '').id
    return render_nb()

@rt
def del_selected():
    if nb.selected is not None:
        i = nb.sel_index()
        nb.remove(nb.selected)
        nb.selected = nb.cells[min(i, len(nb.cells)-1)].id if nb.cells else None
    return render_nb()

@rt
def cut_selected():
    if nb.selected is not None: nb.cut_range(nb.selected)
    return render_nb()

@rt
def copy_selected():
    if nb.selected is not None: nb.copy_range(nb.selected)
    return ''

@rt
def paste_selected():
    nb.paste_after(nb.selected)
    return render_nb()

@rt
def settype_selected(t:str):
    c = nb.get(nb.selected) if nb.selected is not None else None
    if c and t in CTYPES: c.ctype = t
    return render_nb()

@rt
def set_ctype(id:int, t:str):
    "Change one cell's type in place; returns just that cell so the rest of the notebook is untouched."
    c = nb.get(id)
    if c and t in CTYPES and t != c.ctype:
        c.ctype = t
        c.output = None  # stale output no longer meaningful under the new type
    return render_cell(c) if c else render_nb()

# --- inline editing ---
@rt
def edit_cell(id:int):
    c = nb.get(id)
    if not c: return render_nb()
    nb.selected = id
    return render_cell_edit(c)

@rt
def view_cell(id:int):
    c = nb.get(id)
    return render_cell(c) if c else render_nb()

@rt
def save_cell(id:int, source:str):
    c = nb.get(id)
    if c:
        if c.ctype == 'code':
            c.source = _set_export(source, _has_export(c.source))  # editor hides the pragma; preserve the existing flag
            c.output = run_code(c.source)
        elif c.ctype == 'prompt':
            c.source = source
            run_prompt_cell(id)
            return render_nb()
        else:
            c.source = source
    return render_cell(c) if c else render_nb()

@rt
def sync_cell(id:int, source:str):
    "Update a cell's source WITHOUT executing it -- used by Save to flush any editor content that was never explicitly run (Shift+Enter), matching Jupyter's WYSIWYG save behavior."
    c = nb.get(id)
    if c:
        c.source = _set_export(source, _has_export(c.source)) if c.ctype == 'code' else source
    return ''

## Run it

In a notebook, start the server and preview inline. Click a cell's header to
select it, then use command-mode keys: `A`/`B` insert above/below, `D D` delete,
`J`/`K` (or arrows) move selection, `M`/`Y`/`R` change type. In the composer,
`Cmd/Ctrl+/` toggles comments on the selected lines and `Cmd/Ctrl+Shift+-` splits
at the caret. On WSL see the lesson's port notes for reaching it from Windows.

In [ ]:
#| eval: false
srv = JupyUvi(app)
p(index())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()